
## Quarantine Silver Updates
In this lab, you will learn to apply validation rules retrieved from the dataset to bronze tables to create quarantine table, views based on the valid and invalid data.

## Learning Objectives
By the end of this lesson, you should be able to:
- Read rules from  **`look_up_db`** and store them in the form of a dictionary.
- Create a quarantine table by reading a source table and applying the validation rules obtained for deriving valid and invalid views.
- Create validated views using the **`create_validated_views`** function and write configuration information for different datasets.

In [0]:
import dlt
import pyspark.sql.functions as F

## Validate Bronze Tables with Portable Expectations <br>

The steps below include:
- Retrieve rules from a Spark dataset based on a specified topic.
- Use **`spark.conf.get()`** to retrieves the value of the configuration parameter **`"lookup_db"`**.
- Store the rules in python dictionary in form of key value pair 
- Create a quarantined table to mark whether each record should be quarantined or not.
- **Modularize code with views** for valid and invalid data
- Create validated views for that dataset

In [0]:
lookup_db = spark.conf.get("lookup_db")

def get_rules(topic):
    df = spark.read.table(f"{lookup_db}.rules").filter(F.col("topic") == topic)
    rules = {}
    for row in df.collect(): 
        rules[row["name"]] = row["condition"]
    return rules

### Modularize code with views

For this we will use:
- Use **`create_validated_views`** function and pass paramater dataset, source_table, valid_view, invalid_view 
- Use **`get_rules`** function defined earlier to fetch a validation rules and store it in variable.
- Create quarantine table for checking whether data is valid or invalid and include **`is_quarantined`** column to mark whether each record should be quarantined or not. 
- Create function with name **`create_valid`** and apply a filter to select records where "is_quarantined" is false
- Create **`invalid_view`** by applying a filter to select records where "is_quarantined" is true

In [0]:
def create_validated_views(dataset, topic, source_table, valid_view, invalid_view):
    rules = get_rules(topic)
    quarantine_rules = "NOT({0})".format(" AND ".join(rules.values()))
    
    @dlt.table(
        name=f"{dataset}_quarantine",
        # temporary=True, 
        partition_cols=["is_quarantined"]
    )
    @dlt.expect_all(rules)
    def create_quarantine():
        return dlt.read_stream(source_table).withColumn("is_quarantined", F.expr(quarantine_rules))
        
    @dlt.view(name=f"{valid_view}")
    def create_valid():
        return dlt.read_stream(f"{dataset}_quarantine").filter("is_quarantined=false")
    
    @dlt.view(name=f"{invalid_view}")
    def create_invalid():
        return dlt.read_stream(f"{dataset}_quarantine").filter("is_quarantined=true")


### Configuration based code


Use this steps to create configuration:
- Create a dictionary with name **quarantine_tables_config** and include each dataset by a key.
- Use rules_tag, source, valid_view, invalid_view to specify the information of dataset.
- Use **for** loop to iterate through configuration code and each key-value pair in the quarantine_tables_config dictionary.
- Use **`create_validated_views`** function for each dataset by including it inside for loop to separate the data into valid and invalid categories.

In [0]:
quarantine_tables_config = {
    "bpm": { 
      "rules_tag": "bpm",
      "source": "bpm_bronze",
      "valid_view": "valid_bpm",
      "invalid_view": "invalid_bpm"
    },
    "workouts": { 
      "rules_tag": "workout",
      "source": "workouts_bronze",
      "valid_view": "valid_workouts",
      "invalid_view": "invalid_workouts"
    },
    "users_cdc": { 
      "rules_tag": "user_info",
      "source": "users_cdc_bronze",
      "valid_view": "valid_users_cdc",
      "invalid_view": "invalid_users_cdc"
    }
} 

for dataset, c in quarantine_tables_config.items():
    create_validated_views(dataset, c["rules_tag"], c["source"], c["valid_view"], c["invalid_view"])